# Mission 3 ver12 — KLUE-RoBERTa + label attention + supervised contrastive learning

**이 노트북 한 개와 기존 `mission3_ver5_inputs.zip`으로 실행합니다.**
공용 계정의 ZIP 바로가기는 그대로 사용합니다. 기존 ver4 가중치와 보정값은 가져오지 않습니다.
Colab에서 GPU 런타임을 연결하고 **런타임 → 모두 실행**을 누르세요.
Drive 연결 요청에는 ZIP 바로가기가 있는 **공용 계정**을 선택합니다.

- 학습: 서울 Training의 전사 text만. symptom은 9개 정답으로만 사용합니다.
- 예측: 각 증상 2-class head의 `argmax`.
  임계값 검색·보정, 사후 score 이동, 기존 보정 모델 fallback이 없습니다.
- 모델/epoch 선택: 중복 전사가 겹치지 않는 Training 내부 holdout.
- 선택이 끝나면 전체 Training으로 다시 학습하고 공식 Validation을 마지막에 평가합니다.
- 오심·구토를 포함한 증상별 오탐/미탐 CSV와 분포 분석을 자동 저장합니다.

**F1 개선은 실행 결과로 확인해야 합니다.** 기존 0.6567 점수는 threshold 보정을 포함합니다.
새 버전의 Training holdout 지표로 먼저 비교하세요. Validation 점수로 설정을 반복해서 고르지 마세요.
GPU 모델은 holdout 선택 학습과 전체 재학습을 모두 수행하므로 수 시간이 걸릴 수 있습니다. 고정 3시간 제한은 없습니다.


## 가설과 비교 기준
ver11과 같은 encoder·분할·학습률에서 증상별 window attention과 대조학습을 함께 추가합니다.
증상마다 관련 문맥에 가중치를 주고, Training 정답이 같은 증상 있음/없음 표현을 가까이 학습합니다.
같은 통화를 dropout으로 두 번 읽으며 텍스트나 정답을 새로 만들지 않습니다.
두 요소를 함께 바꾸므로 이 비교 하나로 각각의 기여를 확정할 수 없습니다.
오심·구토-only 오탐과 질문/부정 문맥을 Training 오류 CSV에서 확인하세요.
[Supervised Contrastive Learning 논문](https://arxiv.org/abs/2004.11362)


In [ ]:
# @title 실행 설정 (기본값이면 기존 ver9 ZIP 바로가기로 실행)
SETTINGS = {'version': 'ver12', 'backend': 'neural', 'model_id': 'klue/roberta-base', 'revision': '02f94ba5e3fcb7e2a58a390b8639b0fac974a8da', 'label_attention': True, 'contrastive_weight': 0.1, 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver12', 'work_dir': '/content/mission3_categorical', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'sparse_candidates': ['svm_c1', 'svm_c4', 'complement_nb'], 'char_features': 80000, 'word_features': 20000, 'max_length': 512, 'overlap': 64, 'batch_size': 8, 'encoder_window_batch': 4, 'gradient_accumulation': 2, 'max_epochs': 4, 'patience': 2, 'learning_rate': 2e-05, 'head_learning_rate': 0.0001, 'refit_full_training': True, 'allow_cpu': False, 'log_every': 250}
# 직접 JSON 폴더를 쓸 때만 train_dir / validation_dir을 지정하세요.
# max_epochs를 줄이면 모델/epoch 선택 학습 시간은 줄지만 성능이 달라질 수 있습니다.
# refit_full_training=False이면 전체 재학습을 생략하고 80% Training 모델을 사용합니다.
# GPU 메모리 부족 시 batch_size를 절반으로 줄이고 gradient_accumulation을 두 배로 늘립니다.
# 대조학습은 실제 batch 안에서만 비교하므로 accumulation은 비교 표본 수를 늘리지 않습니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비
import subprocess
import sys
packages = ["numpy==2.1.3", "scikit-learn==1.6.1", "scipy>=1.11,<2",
            "iterative-stratification==0.1.9", "joblib>=1.3,<2"]
if SETTINGS["backend"] == "neural":
    packages += ["torch>=2.6,<3", "transformers==4.57.1",
                 "sentencepiece>=0.2,<0.3", "protobuf>=4.25,<7"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])


In [ ]:
# @title 실행 코드 복원 (별도 .py 업로드 불필요)
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Standalone Mission 3 categorical experiments embedded in ver10-ver13."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'neural.py': '"""Text-only dialogue classifiers with ordinary binary class decisions.\n\nEach symptom has two class logits. Predictions use the winning class directly;\nthere is no probability cutoff search, calibration, or post-hoc label weighting.\nPublic pretrained assets are prepared separately from local checkpoint weights.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom pathlib import Path\nimport random\nimport re\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nNUM_LABELS = 9\nCONTRASTIVE_DIM = 64\n_MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")\n\n\ndef seed_everything(seed):\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef prepare_encoder(model_id, revision, asset_dir):\n    """Download an encoder once; store only its config and tokenizer assets.\n\n    The caller stores the trained encoder and head together in its checkpoint.\n    Transformers imports are lazy so CPU tests do not require that dependency.\n    """\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    if not isinstance(model_id, str) or not model_id.strip():\n        raise ValueError("model_id must be a nonempty string")\n    if not isinstance(revision, str) or not revision.strip():\n        raise ValueError("revision must be explicit")\n    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n    resolved = getattr(config, "_commit_hash", None)\n    if resolved is None and re.fullmatch(r"[0-9a-f]{40}", revision):\n        resolved = revision\n    if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-f]{40}", resolved):\n        raise RuntimeError("The immutable pretrained model revision was not resolved")\n    # Resolve a mutable branch exactly once, then pin every other asset to it.\n    options = {"revision": resolved, "trust_remote_code": False}\n    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    encoder = AutoModel.from_pretrained(model_id, config=config, **options)\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    if not getattr(encoder, "supports_gradient_checkpointing", False):\n        raise ValueError("The selected encoder must support gradient checkpointing")\n    encoder.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n    asset_dir = Path(asset_dir)\n    encoder_dir, tokenizer_dir = asset_dir / "encoder", asset_dir / "tokenizer"\n    encoder_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer_dir.mkdir(parents=True, exist_ok=True)\n    encoder.config.save_pretrained(encoder_dir)\n    tokenizer.save_pretrained(tokenizer_dir)\n    return encoder, tokenizer, resolved\n\n\ndef _binary_targets(targets, count):\n    values = torch.as_tensor(targets)\n    if values.shape != (count, NUM_LABELS):\n        raise ValueError(f"targets must have shape ({count}, {NUM_LABELS})")\n    if not bool(torch.isfinite(values).all()) or not bool(((values == 0) | (values == 1)).all()):\n        raise ValueError("targets must contain only finite binary values")\n    return values.to(dtype=torch.long)\n\n\nclass EncodeDataset(Dataset):\n    """Retain all fast-tokenizer overflow windows, including the final tokens."""\n\n    def __init__(self, texts, targets, tokenizer, max_length=512, overlap=64):\n        texts = list(texts)\n        if any(not isinstance(text, str) for text in texts):\n            raise ValueError("Every dialogue must be a text string")\n        for name, value in (("max_length", max_length), ("overlap", overlap)):\n            if isinstance(value, bool) or not isinstance(value, int):\n                raise ValueError(f"{name} must be an integer")\n        usable = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        if usable < 1 or not 0 <= overlap < usable:\n            raise ValueError("max_length must fit special tokens and 0 <= overlap < usable tokens")\n        if getattr(tokenizer, "pad_token_id", None) is None:\n            raise ValueError("The tokenizer must define a padding token")\n        separator = getattr(tokenizer, "sep_token", None)\n        if not isinstance(separator, str) or not separator:\n            raise ValueError("The tokenizer must define an utterance separator token")\n        self.tokenizer = tokenizer\n        self.targets = None if targets is None else _binary_targets(targets, len(texts)).clone()\n        self.documents = [[] for _ in texts]\n        for start in range(0, len(texts), 128):\n            dialogues = [re.sub(r"\\r\\n?|\\n", f" {separator} ", text)\n                         for text in texts[start:start + 128]]\n            encoded = tokenizer(\n                dialogues, truncation=True, max_length=max_length, stride=overlap,\n                return_overflowing_tokens=True, padding=False,\n                return_attention_mask=True,\n            )\n            if "overflow_to_sample_mapping" not in encoded:\n                raise ValueError("A fast tokenizer must return overflow_to_sample_mapping")\n            if "input_ids" not in encoded or "attention_mask" not in encoded:\n                raise ValueError("The tokenizer must return input_ids and attention_mask")\n            for index, source in enumerate(encoded["overflow_to_sample_mapping"]):\n                source = int(source)\n                if not 0 <= source < len(dialogues):\n                    raise ValueError("Tokenizer overflow mapping is out of range")\n                window = {key: list(encoded[key][index])\n                          for key in _MODEL_INPUTS if key in encoded}\n                length = len(window["input_ids"])\n                if not 0 < length <= max_length or any(len(value) != length for value in window.values()):\n                    raise ValueError("Tokenizer returned an invalid window length")\n                if not all(value in (0, 1) for value in window["attention_mask"]) or not any(window["attention_mask"]):\n                    raise ValueError("A real token window must have a binary, nonempty attention mask")\n                self.documents[start + source].append(window)\n        if any(not document for document in self.documents):\n            raise ValueError("Tokenizer returned a dialogue without any windows")\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["targets"] = self.targets[index]\n        return item\n\n    def collate(self, examples):\n        """Pad tokens and windows; keep one target vector for each dialogue."""\n        if not examples or any(not item["windows"] for item in examples):\n            raise ValueError("A batch must contain nonempty dialogue windows")\n        keys = set(examples[0]["windows"][0])\n        if any(set(window) != keys for item in examples for window in item["windows"]):\n            raise ValueError("Token windows must use the same model input fields")\n        if any(("targets" in item) != ("targets" in examples[0]) for item in examples):\n            raise ValueError("A batch cannot mix labeled and unlabeled dialogues")\n        batch_size = len(examples)\n        window_count = max(len(item["windows"]) for item in examples)\n        token_count = max(len(window["input_ids"]) for item in examples for window in item["windows"])\n        batch = {key: torch.full(\n            (batch_size, window_count, token_count),\n            self.tokenizer.pad_token_id if key == "input_ids" else 0,\n            dtype=torch.long,\n        ) for key in _MODEL_INPUTS if key in keys}\n        batch["window_mask"] = torch.zeros((batch_size, window_count), dtype=torch.bool)\n        for row, item in enumerate(examples):\n            for column, window in enumerate(item["windows"]):\n                length = len(window["input_ids"])\n                for key in keys:\n                    batch[key][row, column, :length] = torch.tensor(window[key], dtype=torch.long)\n                batch["window_mask"][row, column] = True\n        if "targets" in examples[0]:\n            batch["targets"] = torch.stack([item["targets"] for item in examples])\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Train nine independent binary heads from all windows of each dialogue."""\n\n    def __init__(self, encoder, label_attention=False, contrastive=False, feature_dim=256):\n        super().__init__()\n        if isinstance(feature_dim, bool) or not isinstance(feature_dim, int) or feature_dim < 1:\n            raise ValueError("feature_dim must be a positive integer")\n        if not isinstance(label_attention, bool) or not isinstance(contrastive, bool):\n            raise ValueError("label_attention and contrastive must be boolean")\n        self.encoder = encoder\n        self.label_attention = label_attention\n        self.contrastive = contrastive\n        self.encoder_window_batch = 8\n        hidden_size = int(encoder.config.hidden_size)\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, feature_dim), nn.GELU())\n        self.dropout = nn.Dropout(0.1)\n        if label_attention:\n            self.token_queries = nn.Parameter(torch.empty(NUM_LABELS, hidden_size))\n            self.window_queries = nn.Parameter(torch.empty(NUM_LABELS, feature_dim))\n            nn.init.normal_(self.token_queries, std=0.02)\n            nn.init.normal_(self.window_queries, std=0.02)\n        else:\n            self.register_parameter("token_queries", None)\n            self.register_parameter("window_queries", None)\n        self.class_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, 2))\n        self.class_bias = nn.Parameter(torch.zeros(NUM_LABELS, 2))\n        self.feature_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, CONTRASTIVE_DIM))\n        self.feature_bias = nn.Parameter(torch.zeros(NUM_LABELS, CONTRASTIVE_DIM))\n        nn.init.xavier_uniform_(self.class_weights)\n        nn.init.xavier_uniform_(self.feature_weights)\n\n    def forward(self, batch):\n        window_mask = batch.get("window_mask")\n        ids, token_mask = batch.get("input_ids"), batch.get("attention_mask")\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 3:\n            raise ValueError("input_ids must have shape (batch, windows, tokens)")\n        if not isinstance(token_mask, torch.Tensor) or token_mask.shape != ids.shape:\n            raise ValueError("attention_mask must match input_ids")\n        if not isinstance(window_mask, torch.Tensor) or window_mask.shape != ids.shape[:2] or window_mask.dtype != torch.bool:\n            raise ValueError("window_mask must be a boolean (batch, windows) tensor")\n        if ids.shape[0] < 1 or not bool(window_mask.any(dim=1).all()):\n            raise ValueError("Every dialogue must contain at least one real window")\n        if not bool(((token_mask == 0) | (token_mask == 1)).all()):\n            raise ValueError("attention_mask must be binary")\n        if not bool((token_mask.sum(-1)[window_mask] > 0).all()):\n            raise ValueError("Real windows must contain attended tokens")\n        chunk_size = self.encoder_window_batch\n        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int) or chunk_size < 1:\n            raise ValueError("encoder_window_batch must be a positive integer")\n        flat_mask = window_mask.flatten()\n        inputs = {}\n        for key in _MODEL_INPUTS:\n            if key in batch:\n                if batch[key].shape != ids.shape:\n                    raise ValueError(f"{key} must match input_ids")\n                inputs[key] = batch[key].flatten(0, 1)[flat_mask]\n        representations = []\n        for start in range(0, len(inputs["input_ids"]), chunk_size):\n            chunk = {key: value[start:start + chunk_size] for key, value in inputs.items()}\n            hidden = self.encoder(**chunk).last_hidden_state\n            if not bool(torch.isfinite(hidden).all()):\n                raise FloatingPointError("Encoder produced nonfinite hidden states")\n            if self.label_attention:\n                token_scores = torch.einsum("nth,lh->ntl", hidden.float(), self.token_queries.float())\n                token_scores = token_scores / math.sqrt(hidden.shape[-1])\n                token_scores = token_scores.masked_fill(~chunk["attention_mask"].bool().unsqueeze(-1), -torch.inf)\n                token_attention = torch.softmax(token_scores, dim=1).to(hidden.dtype)\n                context = torch.einsum("ntl,nth->nlh", token_attention, hidden)\n                cls = hidden[:, 0].unsqueeze(1).expand(-1, NUM_LABELS, -1)\n                representations.append(self.projection(torch.cat((cls, context), dim=-1)))\n            else:\n                mask = chunk["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n                mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n                representations.append(self.projection(torch.cat((hidden[:, 0], mean), dim=-1)))\n        valid = self.dropout(torch.cat(representations, dim=0))\n        if not bool(torch.isfinite(valid).all()):\n            raise FloatingPointError("Window projection produced nonfinite features")\n        padded_shape = (*window_mask.shape, *valid.shape[1:])\n        windows = valid.new_zeros(padded_shape)\n        windows = windows.flatten(0, 1).index_copy(\n            0, flat_mask.nonzero(as_tuple=False).flatten(), valid,\n        ).reshape(padded_shape)\n        if self.label_attention:\n            scores = (windows.float() * self.window_queries.float()[None, None]).sum(-1)\n            scores = scores / math.sqrt(windows.shape[-1])\n            scores = scores.masked_fill(~window_mask.unsqueeze(-1), -torch.inf)\n            attention = torch.softmax(scores, dim=1).to(windows.dtype)\n            pooled = (attention.unsqueeze(-1) * windows).sum(dim=1)\n        else:\n            pooled = windows.sum(dim=1) / window_mask.sum(dim=1, keepdim=True).to(windows.dtype)\n            pooled = pooled.unsqueeze(1).expand(-1, NUM_LABELS, -1)\n        pooled = self.dropout(pooled)\n        logits = torch.einsum("bld,ldc->blc", pooled, self.class_weights) + self.class_bias\n        features = torch.einsum("bld,ldf->blf", pooled, self.feature_weights) + self.feature_bias\n        features = F.normalize(features.float(), dim=-1)\n        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(features).all()):\n            raise FloatingPointError("Classifier produced nonfinite logits or features")\n        return logits, features\n\n\ndef predictions(logits):\n    """The learned head\'s winning class; no fitted post-training cutoff."""\n    if logits.ndim != 3 or logits.shape[1:] != (NUM_LABELS, 2):\n        raise ValueError("logits must have shape (batch, 9, 2)")\n    if not bool(torch.isfinite(logits).all()):\n        raise FloatingPointError("Cannot predict from nonfinite logits")\n    return logits.argmax(dim=-1)\n\n\ndef classification_loss(logits, targets):\n    """Unweighted cross entropy, averaged equally over the nine binary heads."""\n    predictions(logits)\n    targets = _binary_targets(targets, len(logits)).to(logits.device)\n    if len(logits) < 1:\n        raise ValueError("classification_loss requires a nonempty batch")\n    loss = F.cross_entropy(logits.float().reshape(-1, 2), targets.reshape(-1))\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Classification loss is nonfinite")\n    return loss\n\n\ndef binary_supervised_contrastive_loss(features1, features2, targets, temperature=0.1):\n    """Binary supervised contrastive loss per symptom, with balanced anchors.\n\n    The two views come from independent dropout forwards of the same dialogues.\n    Same-class examples are positives, self matches are excluded, and each\n    present binary class receives equal anchor weight within each symptom.\n    Thus the many symptom-negative examples cannot dominate the anchor average.\n    """\n    if features1.ndim != 3 or features1.shape[1] != NUM_LABELS or features1.shape != features2.shape:\n        raise ValueError("Both feature views must have matching (batch, 9, feature_dim) shapes")\n    if features1.shape[0] < 1 or features1.shape[2] < 1:\n        raise ValueError("Contrastive loss requires nonempty feature views")\n    if not bool(torch.isfinite(features1).all()) or not bool(torch.isfinite(features2).all()):\n        raise FloatingPointError("Contrastive features must be finite")\n    if isinstance(temperature, bool) or not math.isfinite(float(temperature)) or temperature <= 0:\n        raise ValueError("temperature must be positive and finite")\n    labels = _binary_targets(targets, len(features1)).to(features1.device)\n    features = F.normalize(torch.cat((features1.float(), features2.float()), dim=0), dim=-1)\n    labels = torch.cat((labels, labels), dim=0)\n    self_mask = torch.eye(len(features), dtype=torch.bool, device=features.device)\n    losses = []\n    for label in range(NUM_LABELS):\n        scores = features[:, label] @ features[:, label].T / float(temperature)\n        denominator = torch.logsumexp(scores.masked_fill(self_mask, -torch.inf), dim=1, keepdim=True)\n        log_probability = scores - denominator\n        positive = (labels[:, label, None] == labels[None, :, label]) & ~self_mask\n        anchor_loss = -(log_probability * positive).sum(1) / positive.sum(1)\n        class_losses = [anchor_loss[labels[:, label] == value].mean()\n                        for value in (0, 1) if bool((labels[:, label] == value).any())]\n        losses.append(torch.stack(class_losses).mean())\n    loss = torch.stack(losses).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Supervised contrastive loss is nonfinite")\n    return loss\n', 'runtime.py': '"""Training-only model selection and categorical Mission 3 inference."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport joblib\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.pipeline import FeatureUnion\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.svm import LinearSVC\n\nfrom . import data\nfrom .bootstrap import prepare_data\n\n\ndef dataset_digest(texts, targets):\n    digest = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(target, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef feature_extractor(settings):\n    return FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                min_df=2, max_features=settings.get("char_features", 80000),\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2,\n                                max_features=settings.get("word_features", 20000),\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n\n\ndef sparse_estimator(name, seed):\n    # These are standard categorical estimators; no score cutoff is searched.\n    if name == "svm_c1":\n        base = LinearSVC(C=1.0, dual="auto", max_iter=5000, random_state=seed)\n    elif name == "svm_c4":\n        base = LinearSVC(C=4.0, dual="auto", max_iter=5000, random_state=seed)\n    elif name == "complement_nb":\n        base = ComplementNB(alpha=1.0)\n    else:\n        raise ValueError(f"Unknown sparse candidate: {name}")\n    return OneVsRestClassifier(base, n_jobs=1)\n\n\ndef train_sparse(paths, texts, targets, fit, holdout, local_dir, output_dir, settings):\n    vectorizer = feature_extractor(settings)\n    train_x = vectorizer.fit_transform([texts[i] for i in fit])\n    holdout_x = vectorizer.transform([texts[i] for i in holdout])\n    candidates = []\n    best = None\n    for name in settings.get("sparse_candidates", ["svm_c1", "svm_c4", "complement_nb"]):\n        started = time.monotonic()\n        estimator = sparse_estimator(name, settings["seed"])\n        estimator.fit(train_x, targets[fit])\n        predictions = estimator.predict(holdout_x).astype(np.int8)\n        report = data.score_predictions(targets[holdout], predictions)\n        candidates.append({"name": name, "seconds": time.monotonic() - started, **report})\n        data.write_json(output_dir / "candidate_metrics.json", candidates)\n        print(f"Training holdout {name}: Macro F1={report[\'macro_f1\']:.6f}", flush=True)\n        if best is None or report["macro_f1"] > best[0]:\n            best = (report["macro_f1"], name, predictions, report)\n    if best is None:\n        raise ValueError("At least one sparse candidate is required")\n    np.savez_compressed(output_dir / "holdout_predictions.npz", indices=holdout,\n                        targets=targets[holdout], predictions=best[2])\n    data.export_errors([paths[i] for i in holdout],\n                       [texts[i] for i in holdout], targets[holdout], best[2],\n                       output_dir / "training_errors")\n    del train_x, holdout_x, estimator, vectorizer\n    gc.collect()\n    print(f"Selected using Training only: {best[1]}; refitting all Training", flush=True)\n    vectorizer = feature_extractor(settings)\n    full_x = vectorizer.fit_transform(texts)\n    estimator = sparse_estimator(best[1], settings["seed"])\n    estimator.fit(full_x, targets)\n    joblib.dump({"vectorizer": vectorizer, "estimator": estimator}, local_dir / "sparse.joblib", compress=3)\n    shutil.copy2(local_dir / "sparse.joblib", output_dir / "sparse.joblib")\n    manifest = {"kind": "sparse", "artifact": "sparse.joblib", "selected_name": best[1]}\n    return manifest, best[3]\n\n\ndef _save_weights(model, path):\n    import torch\n    temporary = path.with_name(path.name + ".tmp")\n    # CPU tensors make the saved checkpoint device-independent.\n    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, temporary)\n    temporary.replace(path)\n\n\ndef _load_weights(model, path):\n    import torch\n    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)\n\n\ndef _loader(texts, targets, tokenizer, settings, shuffle):\n    from torch.utils.data import DataLoader\n    from .neural import EncodeDataset\n    dataset = EncodeDataset(texts, targets, tokenizer,\n                            max_length=settings["max_length"], overlap=settings["overlap"])\n    return DataLoader(dataset, batch_size=settings["batch_size"], shuffle=shuffle,\n                      num_workers=0, collate_fn=dataset.collate)\n\n\ndef _to_device(batch, device):\n    return {key: value.to(device) for key, value in batch.items()}\n\n\ndef _predict(model, loader, device):\n    import torch\n    from .neural import predictions\n    result = []\n    model.eval()\n    with torch.inference_mode():\n        for batch in loader:\n            logits, _ = model(_to_device(batch, device))\n            result.append(predictions(logits).cpu().numpy())\n    if not result:\n        raise ValueError("Cannot predict an empty dataset")\n    return np.concatenate(result).astype(np.int8)\n\n\ndef _fit_epochs(model, loader, settings, device, output_dir, prefix,\n                holdout_loader=None, holdout_targets=None, fixed_epochs=None):\n    import torch\n    from .neural import classification_loss, binary_supervised_contrastive_loss\n    encoder_parameters = list(model.encoder.parameters())\n    encoder_ids = {id(parameter) for parameter in encoder_parameters}\n    head_parameters = [p for p in model.parameters() if id(p) not in encoder_ids]\n    optimizer = torch.optim.AdamW([\n        {"params": encoder_parameters, "lr": settings["learning_rate"]},\n        {"params": head_parameters, "lr": settings["head_learning_rate"]},\n    ], weight_decay=0.01)\n    epochs = fixed_epochs or settings["max_epochs"]\n    accumulation = settings["gradient_accumulation"]\n    updates_per_epoch = (len(loader) + accumulation - 1) // accumulation\n    total_updates = updates_per_epoch * epochs\n    warmup = max(1, int(total_updates * 0.1))\n\n    def schedule(step):\n        if step < warmup:\n            return (step + 1) / warmup\n        return max(0.0, (total_updates - step) / max(1, total_updates - warmup))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    use_cuda = device.type == "cuda"\n    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16\n    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda and dtype == torch.float16)\n    best_f1, best_epoch, stale = -1.0, 0, 0\n    history = []\n    weight_path = output_dir / f"{prefix}_weights.pt"\n    for epoch in range(1, epochs + 1):\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        total_loss = 0.0\n        for step, batch in enumerate(loader):\n            batch = _to_device(batch, device)\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_cuda):\n                logits, features1 = model(batch)\n                loss = classification_loss(logits, batch["targets"])\n                if settings["contrastive_weight"]:\n                    logits2, features2 = model(batch)\n                    ce2 = classification_loss(logits2, batch["targets"])\n                    loss = (loss + ce2) / 2 + settings["contrastive_weight"] * (\n                        binary_supervised_contrastive_loss(features1, features2, batch["targets"]))\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite training loss; no checkpoint was accepted")\n            group_start = (step // accumulation) * accumulation\n            group_size = min(accumulation, len(loader) - group_start)\n            scaler.scale(loss / group_size).backward()\n            total_loss += float(loss.detach())\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n                scaler.step(optimizer)\n                scaler.update()\n                scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            if (step + 1) % settings.get("log_every", 250) == 0:\n                print(f"{prefix} epoch {epoch}: {step + 1}/{len(loader)} batches", flush=True)\n        row = {"epoch": epoch, "loss": total_loss / len(loader),\n               "seconds": time.monotonic() - started}\n        if holdout_loader is not None:\n            predicted = _predict(model, holdout_loader, device)\n            metrics = data.score_predictions(holdout_targets, predicted)\n            row["holdout_macro_f1"] = metrics["macro_f1"]\n            if metrics["macro_f1"] > best_f1:\n                best_f1, best_epoch, stale = metrics["macro_f1"], epoch, 0\n                _save_weights(model, weight_path)\n                data.write_json(output_dir / "selected_holdout_metrics.json", metrics)\n            else:\n                stale += 1\n            print(f"Training holdout epoch {epoch}: Macro F1={metrics[\'macro_f1\']:.6f}", flush=True)\n        else:\n            _save_weights(model, weight_path)\n            best_epoch = epoch\n            print(f"All-Training refit epoch {epoch}/{epochs}: loss={row[\'loss\']:.5f}", flush=True)\n        history.append(row)\n        data.write_json(output_dir / f"{prefix}_history.json", history)\n        if holdout_loader is not None and stale >= settings["patience"]:\n            break\n    return best_epoch, weight_path\n\n\ndef train_neural(paths, texts, targets, fit, holdout, local_dir, output_dir, settings):\n    import torch\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything\n    if not torch.cuda.is_available() and not settings.get("allow_cpu", False):\n        raise RuntimeError("This version needs a Colab GPU runtime; ver10 runs on CPU")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    seed_everything(settings["seed"])\n    asset_dir = output_dir / "assets"\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], asset_dir)\n    settings = {**settings, "resolved_revision": revision}\n    data.write_json(output_dir / "settings.json", settings)\n    model = DialogueClassifier(encoder, label_attention=settings["label_attention"],\n                               contrastive=bool(settings["contrastive_weight"]))\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    fit_loader = _loader([texts[i] for i in fit], targets[fit], tokenizer, settings, True)\n    holdout_loader = _loader([texts[i] for i in holdout], targets[holdout], tokenizer, settings, False)\n    best_epoch, selected_path = _fit_epochs(model, fit_loader, settings, device, output_dir,\n                                           "selection", holdout_loader, targets[holdout])\n    _load_weights(model, selected_path)\n    predicted = _predict(model, holdout_loader, device)\n    metrics = data.score_predictions(targets[holdout], predicted)\n    np.savez_compressed(output_dir / "holdout_predictions.npz", indices=holdout,\n                        targets=targets[holdout], predictions=predicted)\n    data.export_errors([paths[i] for i in holdout], [texts[i] for i in holdout],\n                       targets[holdout], predicted, output_dir / "training_errors")\n    if settings.get("refit_full_training", True):\n        del model, encoder, fit_loader, holdout_loader\n        gc.collect()\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n        seed_everything(settings["seed"])\n        # Fresh public encoder; selected weights do not seed the refit.\n        encoder, tokenizer, _ = prepare_encoder(settings["model_id"], revision, asset_dir)\n        model = DialogueClassifier(encoder, label_attention=settings["label_attention"],\n                                   contrastive=bool(settings["contrastive_weight"]))\n        model.encoder_window_batch = settings["encoder_window_batch"]\n        model.to(device)\n        full_loader = _loader(texts, targets, tokenizer, settings, True)\n        _, final_weights = _fit_epochs(model, full_loader, settings, device, output_dir,\n                                       "final", fixed_epochs=best_epoch)\n        fitted_samples = len(targets)\n    else:\n        final_weights = selected_path\n        fitted_samples = len(fit)\n    # A final artifact exists only once its declared refit has completed.\n    manifest = {"kind": "neural", "weights": final_weights.name, "assets": "assets",\n                "model_id": settings["model_id"], "resolved_revision": revision,\n                "selected_epochs": best_epoch, "fitted_samples": fitted_samples,\n                "label_attention": settings["label_attention"],\n                "contrastive": bool(settings["contrastive_weight"]),\n                "max_length": settings["max_length"], "overlap": settings["overlap"],\n                "batch_size": settings["batch_size"], "encoder_window_batch": settings["encoder_window_batch"]}\n    del model, encoder\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return manifest, metrics\n\n\ndef _relative_artifact(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("Model artifact path escapes checkpoint directory")\n    return path\n\n\ndef predict_checkpoint(checkpoint, texts):\n    checkpoint = Path(checkpoint)\n    manifest = json.loads(checkpoint.read_text(encoding="utf-8-sig"))\n    if manifest.get("format") != "mission3_categorical_v1" or tuple(manifest["labels"]) != data.TARGET_SYMPTOMS:\n        raise ValueError("Unsupported checkpoint or symptom ordering")\n    root = checkpoint.parent\n    if manifest["kind"] == "sparse":\n        bundle = joblib.load(_relative_artifact(root, manifest["artifact"]))\n        return bundle["estimator"].predict(bundle["vectorizer"].transform(texts)).astype(np.int8)\n    if manifest["kind"] != "neural":\n        raise ValueError("Unknown checkpoint model kind")\n    import torch\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n    from .neural import DialogueClassifier\n    assets = _relative_artifact(root, manifest["assets"])\n    config = AutoConfig.from_pretrained(assets / "encoder", local_files_only=True, trust_remote_code=False)\n    encoder = AutoModel.from_config(config, trust_remote_code=False)\n    tokenizer = AutoTokenizer.from_pretrained(assets / "tokenizer", local_files_only=True, trust_remote_code=False)\n    model = DialogueClassifier(encoder, label_attention=manifest["label_attention"],\n                               contrastive=manifest["contrastive"])\n    model.encoder_window_batch = manifest["encoder_window_batch"]\n    _load_weights(model, _relative_artifact(root, manifest["weights"]))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device)\n    loader = _loader(texts, None, tokenizer, manifest, False)\n    return _predict(model, loader, device)\n\n\ndef inference(checkpoint, label_dir, output):\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predicted = predict_checkpoint(checkpoint, texts)\n    data.write_predictions(paths, predicted, output)\n    return paths, predicted\n\n\ndef run(settings, runtime_sources=None):\n    started = time.monotonic()\n    settings = dict(settings)\n    train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n                                             settings.get("train_dir", ""), settings.get("validation_dir", ""))\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    output_dir = Path(settings["output_dir"]) / run_id\n    output_dir.mkdir(parents=True, exist_ok=False)\n    local_dir = Path(settings["work_dir"]) / run_id\n    local_dir.mkdir(parents=True, exist_ok=False)\n    data.write_json(output_dir / "settings.json", settings)\n    if runtime_sources:\n        submission = output_dir / "submission"\n        for name, source in runtime_sources.items():\n            if name.startswith("test_") or name == "build_notebooks.py":\n                continue\n            target = submission / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_text(source, encoding="utf-8")\n    paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n    fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n    data.write_json(output_dir / "training_diagnostics.json", data.training_diagnostics(texts, targets))\n    np.savez_compressed(output_dir / "training_split.npz", fit=fit, holdout=holdout)\n    data.write_json(output_dir / "run_identity.json", {\n        "version": settings["version"], "training_digest": dataset_digest(texts, targets),\n        "training_samples": len(texts), "fit_samples": len(fit), "holdout_samples": len(holdout),\n        "model_selection": "Training internal holdout only", "decision_rule": "categorical predict/argmax",\n    })\n    print(f"Training {len(texts):,}; fit {len(fit):,}; holdout {len(holdout):,}", flush=True)\n    print("Training label counts:", dict(zip(data.TARGET_SYMPTOMS, targets.sum(axis=0).tolist())), flush=True)\n    try:\n        trainer = train_sparse if settings["backend"] == "sparse" else train_neural\n        manifest, metrics = trainer(paths, texts, targets, fit, holdout, local_dir, output_dir, settings)\n        manifest.update(format="mission3_categorical_v1", labels=list(data.TARGET_SYMPTOMS),\n                        version=settings["version"], decision_rule="categorical predict/argmax")\n        checkpoint = output_dir / f"mission3_{settings[\'version\']}_model.ckpt"\n        data.write_json(output_dir / "training_metrics.json", metrics)\n        data.write_json(checkpoint, manifest)\n        # Reopen the serialized model on the holdout text before official inference.\n        probe = [texts[i] for i in holdout[:min(12, len(holdout))]]\n        roundtrip = predict_checkpoint(checkpoint, probe)\n        if roundtrip.shape != (len(probe), len(data.TARGET_SYMPTOMS)):\n            raise ValueError("Checkpoint roundtrip shape mismatch")\n        if settings.get("run_validation", True):\n            validation_paths, predicted = inference(checkpoint, validation_dir, output_dir / "mission3.csv")\n            # Official targets are accessed only after predictions are saved.\n            scored_paths, _, validation_targets = data.load_dataset(validation_dir, include_targets=True)\n            if [p.name for p in scored_paths] != [p.name for p in validation_paths]:\n                raise ValueError("Official evaluation file order mismatch")\n            report = data.score_predictions(validation_targets, predicted)\n            data.write_json(output_dir / "validation_metrics.json", report)\n            print(f"Official Validation (final diagnostic): Macro F1={report[\'macro_f1\']:.6f}", flush=True)\n            for name, value in report["per_symptom"].items():\n                print(f"  {name}: F1={value[\'f1\']:.4f}, P={value[\'precision\']:.4f}, R={value[\'recall\']:.4f}", flush=True)\n        data.write_json(output_dir / "runtime.json", {"status": "complete", "seconds": time.monotonic() - started})\n    except Exception as error:\n        data.write_json(output_dir / "runtime.json", {"status": "failed", "seconds": time.monotonic() - started,\n                                                      "error_type": type(error).__name__})\n        raise\n    print("Saved results:", output_dir, flush=True)\n    return output_dir\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", required=True)  # Official interface; Mission 3 uses text only.\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    inference(args.ckpt_path, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""Official one-command inference entry point for a portable result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_categorical_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Run after dependency installation in a fresh interpreter, avoiding stale ABI imports."""\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args()\n    root = Path(__file__).resolve().parent\n    name = "mission3_categorical_execution"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    runtime = importlib.import_module(name + ".runtime")\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n    sources = {path.name: path.read_text(encoding="utf-8") for path in root.iterdir()\n               if path.is_file() and (path.suffix in (".py", ".md") or path.name == "requirements.txt")}\n    result = runtime.run(settings, runtime_sources=sources)\n    Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': '# Mission 3 ver10–ver13 실행 안내\n\n네 노트북은 기존 `mission3_ver5_inputs.zip`과 호환되는 독립 Colab 파일입니다.\n다른 Python 파일을 업로드하지 않아도 됩니다. 기존 ver1–ver9는 수정하지 않습니다.\n\n## 실행\n\n1. ZIP 바로가기가 있는 공용 계정으로 Colab에 로그인합니다.\n2. `mission3_ver10.ipynb`부터 열어 `런타임 → 모두 실행`을 선택합니다.\n3. ver11–ver13은 GPU 런타임이 필요합니다. 기본 배치는 ver11=4, ver12–13=8입니다.\n   메모리가 부족하면 batch_size를 절반으로 줄이고 gradient_accumulation을 두 배로 늘립니다.\n4. 결과는 `MyDrive/DCC_AIdeas/results/mission3/ver번호/실행시각_고유번호/`에 저장됩니다.\n\n| 파일 | 모델 | 비교 목적 |\n| --- | --- | --- |\n| mission3_ver10.ipynb | 문자·단어 TF-IDF + LinearSVC / ComplementNB | 보정 없는 빠른 분류 기준 |\n| mission3_ver11.ipynb | KLUE-RoBERTa-base + 9개 이진 categorical head | 표준 교차엔트로피 기준 |\n| mission3_ver12.ipynb | 같은 RoBERTa + 증상 attention + 대조학습 | 증상별 문맥과 있음/없음 표현 구별 |\n| mission3_ver13.ipynb | KcELECTRA v2022 + 같은 attention·대조학습 | 구어체 사전학습 encoder 비교 |\n\nGPU 버전은 내부 선택 학습 뒤 선택된 epoch 수만큼 공개 encoder에서 새로 전체 Training을\n학습하므로 수 시간이 걸릴 수 있습니다. 이전 버전의 3시간 예산 제한은 적용하지 않습니다.\n`refit_full_training=False`는 전체 재학습을 생략하지만 Training의 80%로 학습한 모델이 최종 모델이 됩니다.\n중단 시 완료된 epoch 가중치·로그는 남으며, 자동 재개 기능은 없습니다. 재실행은 새 결과 폴더를 만듭니다.\n\n## 규칙 적용\n\n- 입력은 `utterances[].text`만 사용합니다. 발화자·시간·성별·주소·symptom을 특징에 넣지 않습니다.\n- symptom은 Training 정답과 최종 평가 정답으로만 사용합니다.\n- 9개 외 정답만 제거하고 통화 자체는 유지합니다. 0개 또는 여러 증상 예측이 가능합니다.\n- 공개 pretrained encoder만 활용하며 추가 학습 corpus나 상용 AI API를 사용하지 않습니다.\n- 학습·epoch·후보 선택은 Training 내부 분할에서만 수행합니다. 같은 전사는 한쪽 집합에만 둡니다.\n- TF-IDF 어휘는 내부 fit 집합에서만 학습하고, 최종 선택 후 전체 Training에서 다시 학습합니다.\n- 분류기의 `predict`, 신경망의 증상별 두 class logits의 `argmax`를 그대로 사용합니다.\n  threshold 검색·calibration·사후 score 이동·증상별 앙상블 가중치·기존 보정 모델 fallback은 없습니다.\n- argmax도 수학적 결정 경계를 갖습니다. 여기서 제거한 것은 사후 임계값 선택·튜닝입니다.\n  멘토님의 지시를 따르며 이를 “수학적으로 경계가 전혀 없는 모델”이라고 해석하지 않습니다.\n- 공식 Validation은 최종 예측 CSV 저장 뒤 평가만 합니다. 여러 버전 선택에는 Training 지표를 사용합니다.\n\n출제 조건은 `docs/0820_DCC 대학부_예선_출제문제.pdf` 5·9–12쪽에서 확인했습니다.\n해당 PDF에 threshold 금지 문구는 없지만 전달받은 멘토 지침을 추가 조건으로 적용했습니다.\n\n## 오심 오류를 먼저 보는 이유\n\n저장된 ver9 공식 Validation의 오심 F1은 0.4031, 정밀도 0.3205, 재현율 0.5431,\n오탐 494개, 미탐 196개입니다. 기존 최종 예측은 새 DeBERTa보다 기존 ver4 fallback의 결과입니다.\nTraining calibration의 기존 보정 모델은 구토만 정답인 311개 중 143개를 오심으로 예측했습니다.\n이는 최적화에 사용된 자료의 기술적 분석이며 독립적인 평가 점수가 아닙니다.\n\nTraining 전체 29,200개에서 오심 정답은 3,341개, 구토 정답은 4,463개입니다.\n넓은 오심 표현 정규식으로도 오심 정답 2,901개(86.8%)는 직접적인 표현을 찾지 못했습니다.\n따라서 키워드만으로 라벨을 바꾸거나 샘플을 제거하지 않습니다. 자동 표현 집계는 원인 확정 자료가 아닙니다.\n\n각 실행은 `training_diagnostics.json`, `training_errors/errors_오심.csv` 등을 만듭니다.\n오탐과 미탐에서 질문과 실제 답변, 부정 표현, 구토만 있는 경우, 간접 증상 표현을 사람이 검토합니다.\nCSV는 실제 전사를 포함하고 화면에는 집계 지표만 출력합니다.\n\n## 대조학습\n\nver12–ver13은 동일한 Training 통화를 두 번 dropout forward해 두 표현을 얻습니다.\n증상별 있음/없음 정답이 같은 다른 표현을 가까이 하고 다른 class를 구별하는 손실을 교차엔트로피에 추가합니다.\n각 증상의 양성·음성 anchor 평균을 균등하게 집계하여 다수 음성이 대조학습 항을 지배하지 않도록 합니다.\n대조학습은 실제 batch의 통화끼리만 비교합니다. gradient accumulation은 비교집합을 늘리지 않습니다.\n불확실한 정답을 자동 수정하지 않습니다. 효과는 Training holdout의 증상별 P/R/F1로 확인해야 합니다.\nver11→ver12는 attention과 손실을 함께 바꾸므로 각각의 효과를 분리한 인과적 실험은 아닙니다.\n\n## 결과와 제출\n\n`.ckpt`는 manifest입니다. 동일 실행 폴더의 `sparse.joblib` 또는 `.pt`와 `assets`를 함께 보관하세요.\n최종 모델은 재로딩해 확인한 뒤 Validation CSV를 생성합니다. 새 환경 추론은 모델 다운로드가 필요 없습니다.\n\n```text\npython submission/inference.py --audio_dir /path/audio --label_dir /path/json --ckpt_path mission3_ver11_model.ckpt --output mission3.csv\n```\n\nCSV 헤더는 `label file name,symptom`이며 기존 string-list 형식을 유지합니다.\n학습 로그가 저장된 노트북, 실행 코드, 가중치, requirements, 실제 결과 PPT를 제출 규격대로 준비합니다.\n실제 F1 향상과 Colab GPU 실행 시간은 GPU 학습 전에는 검증되지 않습니다.\n기존 약 0.6567은 사후 threshold 보정이 포함되어 새 규칙 기준과 동일한 baseline이 아닙니다.\n\n## “co” 힌트와 자료\n\n모델명은 확정할 수 없습니다. 한국어 “코”였다면 KoBERT도 후보입니다. ComplementNB나\ncontrastive learning을 멘토님이 말한 모델이라고 단정하지 않습니다.\nKoBERT는 공식 전용 tokenizer wrapper가 필요해 단순 AutoTokenizer 비교군에서 제외했습니다.\n\n- [KLUE 공식 모델](https://huggingface.co/klue/roberta-base)\n- [KcELECTRA 작성자 모델 카드](https://huggingface.co/beomi/KcELECTRA-base)\n- [SKT 공식 KoBERT](https://github.com/SKTBrain/KoBERT)\n- [Supervised Contrastive Learning 논문](https://arxiv.org/abs/2004.11362)\n- [ComplementNB 공식 설명](https://scikit-learn.org/1.6/modules/naive_bayes.html#complement-naive-bayes)\n\n공식 Hugging Face API로 확인한 공개 버전을 고정했습니다.\nKLUE-RoBERTa-base: `02f94ba5e3fcb7e2a58a390b8639b0fac974a8da`.\nKcELECTRA v2022: `4cf9eb1dfeec0a672afe73059882fc998a299ceb`.\n\n생성 노트북은 `build_notebooks.py`에서 검증된 실행 코드를 모두 포함해 만듭니다.\n기존 프로젝트와 같은 dependency 범위를 사용하며 루트 requirements는 변경하지 않습니다.\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_categorical_"))
for name, source in RUNTIME_SOURCES.items():
    (runtime_dir / name).write_text(source, encoding="utf-8")
print("실행 코드 준비 완료")


In [ ]:
# @title Training 분석 → 내부 비교 → 최종 학습 → CSV → 마지막 평가
import json
settings_file = runtime_dir / "notebook_settings.json"
result_file = runtime_dir / "notebook_result.json"
settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False), encoding="utf-8")
# 설치 전 커널에 로드된 라이브러리와 섞이지 않게 새 실행 환경을 사용합니다.
subprocess.check_call([sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
                       "--settings", str(settings_file), "--result_file", str(result_file)])
result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))


In [ ]:
import json
for filename in ["training_metrics.json", "validation_metrics.json", "runtime.json"]:
    path = result_dir / filename
    if path.is_file():
        report = json.loads(path.read_text(encoding="utf-8-sig"))
        if "macro_f1" in report:
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
        else:
            print("소요 시간:", round(report["seconds"] / 60, 1), "분")
print("결과 폴더:", result_dir)
print("예측 CSV:", result_dir / "mission3.csv")
print("Training 오탐·미탐:", result_dir / "training_errors")


## 결과와 제출
각 실행은 `ver12/실행시각_고유번호`에 저장되어 기존 결과를 덮어쓰지 않습니다.
`.ckpt`는 manifest이므로 **같은 실행 폴더 전체**의 가중치와 assets도 함께 보관하세요.
실행 로그가 포함된 이 노트북을 저장합니다. 제출 추론 명령:

```bash
python submission/inference.py --audio_dir /path/audio --label_dir /path/json --ckpt_path mission3_ver12_model.ckpt --output mission3.csv
```

추론은 다운로드가 필요 없는 저장된 모델로 실행합니다. CSV 헤더는 `label file name,symptom`입니다.
`training_errors` CSV는 실제 통화 전사를 포함하므로 모델 오류 검토에만 사용하세요.
질문·부정 표현 집계와 검토 표의 후보 표시는 라벨 오류의 확정 근거가 아닙니다.
문제에서 요구하는 결과 PPT는 실제 학습·평가를 마친 뒤 작성해야 합니다.
